In [23]:
import json
import pandas as pd

TEAM2_JSON_PATH = "team2_output.json"

with open(
    TEAM2_JSON_PATH,
    "r",
    encoding="utf-8-sig",
) as file:
    team2_output = json.load(file)

print("최상위 데이터 타입:", type(team2_output))
print("JSON 데이터 개수:", len(team2_output))

team2_df = pd.DataFrame(team2_output)

print("DataFrame 크기:", team2_df.shape)
print("컬럼 목록:")
print(team2_df.columns.tolist())

display(team2_df.head())

최상위 데이터 타입: <class 'list'>
JSON 데이터 개수: 121
DataFrame 크기: (121, 11)
컬럼 목록:
['cve_id', 'cvss_score', 'severity', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe', 'description', 'service', 'version']


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description,service,version
0,CVE-2011-2523,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable...",ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,The client in OpenSSH before 7.2 mishandles fa...,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,NETWORK,HIGH,NONE,NONE,CWE-264,The kbdint_next_device function in auth2-chall...,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,LOCAL,LOW,LOW,NONE,CWE-1262,The do_setup_env function in session.c in sshd...,ssh,4.7p1 Debian 8ubuntu1


In [24]:
# ==========================================================
# 팀2 JSON을 불러오고 기존 학습 모델을 준비
# ==========================================================

import json
import pandas as pd
import joblib

from scipy.sparse import csr_matrix, hstack


TEAM2_JSON_PATH = "team2_output.json"
MODEL_PATH = "risk_model.pkl"


# 팀2 정제 JSON 불러오기
with open(
    TEAM2_JSON_PATH,
    "r",
    encoding="utf-8-sig",
) as file:
    team2_output = json.load(file)


# JSON 최상위 구조 검증
if not isinstance(team2_output, list):
    raise ValueError(
        "팀2 JSON의 최상위 구조는 list 형식이어야 합니다."
    )

if not team2_output:
    raise ValueError(
        "팀2 JSON에 데이터가 없습니다."
    )


# list[dict] → DataFrame
df = pd.DataFrame(team2_output)


# 기존 약 1만 건으로 학습한 Random Forest 모델 불러오기
model = joblib.load(MODEL_PATH)


print("팀2 입력 파일:", TEAM2_JSON_PATH)
print("팀2 입력 데이터 크기:", df.shape)
print("학습 모델:", MODEL_PATH)
print("입력 컬럼:", df.columns.tolist())

display(df.head())

팀2 입력 파일: team2_output.json
팀2 입력 데이터 크기: (121, 11)
학습 모델: risk_model.pkl
입력 컬럼: ['cve_id', 'cvss_score', 'severity', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe', 'description', 'service', 'version']


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description,service,version
0,CVE-2011-2523,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable...",ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,The client in OpenSSH before 7.2 mishandles fa...,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,NETWORK,HIGH,NONE,NONE,CWE-264,The kbdint_next_device function in auth2-chall...,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,LOCAL,LOW,LOW,NONE,CWE-1262,The do_setup_env function in session.c in sshd...,ssh,4.7p1 Debian 8ubuntu1


In [25]:
# ==========================================================
# 2. 모델 예측에 필요한 필수 컬럼 확인
# ==========================================================

required_columns = [
    "cve_id",
    "cvss_score",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:
    raise ValueError(
        f"팀2 JSON에 필요한 컬럼이 없습니다: {missing_columns}"
    )


print("필수 컬럼 확인 완료")

필수 컬럼 확인 완료


In [26]:
# ==========================================================
# 3. 팀2 JSON 문자열 및 UNKNOWN 값 정리
# ==========================================================

# 범주형 컬럼 목록
categorical_columns = [
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
]


# ----------------------------------------------------------
# 3-1. 범주형 컬럼 정리
#
# 처리 내용:
# 1. 실제 null 값이 남아 있으면 UNKNOWN으로 변경
# 2. 숫자 등 다른 타입이 있어도 문자열로 변환
# 3. 앞뒤 공백 제거
# 4. 소문자를 대문자로 통일
# ----------------------------------------------------------
for column in categorical_columns:
    df[column] = (
        df[column]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )


# ----------------------------------------------------------
# 3-2. 문자열 형태의 결측값도 UNKNOWN으로 변경
#
# JSON 정제 과정에서 값이 실제 null이 아니라
# "null", "none", "nan", "" 문자열로 남을 수도 있음
# ----------------------------------------------------------
invalid_values = [
    "",
    "NULL",
    "NONE",
    "NAN",
    "<NA>",
]


for column in categorical_columns:
    df[column] = df[column].replace(
        invalid_values,
        "UNKNOWN",
    )


# ----------------------------------------------------------
# 3-3. description 정리
#
# 설명이 없으면 빈 문자열로 처리
# TF-IDF는 빈 문자열도 오류 없이 transform 가능
# ----------------------------------------------------------
df["description"] = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# 문자열로 들어간 null 표현 정리
df["description"] = df["description"].replace(
    [
        "NULL",
        "null",
        "NONE",
        "None",
        "NAN",
        "nan",
        "<NA>",
    ],
    "",
)


# ----------------------------------------------------------
# 3-4. CVE ID 정리
# ----------------------------------------------------------
df["cve_id"] = (
    df["cve_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# ----------------------------------------------------------
# 3-5. CVSS 점수를 숫자형으로 변환
#
# 숫자가 아닌 값은 NaN으로 만든 뒤 0.0으로 처리
# ----------------------------------------------------------
df["cvss_score"] = pd.to_numeric(
    df["cvss_score"],
    errors="coerce",
).fillna(0.0)


# CVSS 정상 범위인 0~10으로 제한
df["cvss_score"] = df["cvss_score"].clip(
    lower=0.0,
    upper=10.0,
)


# ----------------------------------------------------------
# 3-6. service와 version은 모델 Feature는 아니지만
# 최종 챗봇 출력용으로 정리
# ----------------------------------------------------------
for column in [
    "service",
    "version",
]:
    if column in df.columns:
        df[column] = (
            df[column]
            .fillna("UNKNOWN")
            .astype(str)
            .str.strip()
        )

        df[column] = df[column].replace(
            [
                "",
                "null",
                "NULL",
                "None",
                "NONE",
                "nan",
                "NAN",
                "<NA>",
            ],
            "UNKNOWN",
        )


# ----------------------------------------------------------
# 3-7. 기존 severity도 있다면 정리
# ----------------------------------------------------------
if "severity" in df.columns:
    df["severity"] = (
        df["severity"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df["severity"] = df["severity"].replace(
        invalid_values,
        "UNKNOWN",
    )


print("문자열 및 UNKNOWN 처리 완료")
print("정리 후 데이터 크기:", df.shape)

display(df.head())

문자열 및 UNKNOWN 처리 완료
정리 후 데이터 크기: (121, 11)


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description,service,version
0,CVE-2011-2523,9.8,CRITICAL,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable...",ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-287,The client in OpenSSH before 7.2 mishandles fa...,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,NETWORK,HIGH,UNKNOWN,UNKNOWN,CWE-264,The kbdint_next_device function in auth2-chall...,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,LOCAL,LOW,LOW,UNKNOWN,CWE-1262,The do_setup_env function in session.c in sshd...,ssh,4.7p1 Debian 8ubuntu1


In [27]:
# ==========================================================
# 4. 정제 결과 검증
# ==========================================================

print("전체 데이터 수:", len(df))


print("\n[컬럼별 실제 null 개수]")
print(
    df[
        [
            "cve_id",
            "cvss_score",
            "attack_vector",
            "attack_complexity",
            "privileges_required",
            "user_interaction",
            "cwe",
            "description",
        ]
    ].isnull().sum()
)


print("\n[범주형 컬럼별 UNKNOWN 개수]")

for column in categorical_columns:
    unknown_count = (
        df[column] == "UNKNOWN"
    ).sum()

    print(
        f"{column}: {unknown_count}개"
    )


print("\n[범주형 컬럼별 고유값]")

for column in categorical_columns:
    print(
        f"\n{column}:"
    )

    print(
        sorted(
            df[column].unique().tolist()
        )
    )

전체 데이터 수: 121

[컬럼별 실제 null 개수]
cve_id                 0
cvss_score             0
attack_vector          0
attack_complexity      0
privileges_required    0
user_interaction       0
cwe                    0
description            0
dtype: int64

[범주형 컬럼별 UNKNOWN 개수]
attack_vector: 0개
attack_complexity: 18개
privileges_required: 109개
user_interaction: 119개
cwe: 41개

[범주형 컬럼별 고유값]

attack_vector:
['LOCAL', 'NETWORK']

attack_complexity:
['HIGH', 'LOW', 'MEDIUM', 'UNKNOWN']

privileges_required:
['HIGH', 'LOW', 'SINGLE', 'UNKNOWN']

user_interaction:
['REQUIRED', 'UNKNOWN']

cwe:
['CWE-116', 'CWE-119', 'CWE-1262', 'CWE-16', 'CWE-189', 'CWE-19', 'CWE-20', 'CWE-200', 'CWE-203', 'CWE-22', 'CWE-252', 'CWE-255', 'CWE-264', 'CWE-287', 'CWE-295', 'CWE-352', 'CWE-362', 'CWE-399', 'CWE-400', 'CWE-426', 'CWE-476', 'CWE-667', 'CWE-732', 'CWE-74', 'CWE-77', 'CWE-770', 'CWE-776', 'CWE-78', 'CWE-787', 'CWE-79', 'CWE-863', 'CWE-89', 'CWE-93', 'UNKNOWN']


In [28]:
# ==========================================================
# 5. 허용 범주 이외 값 검사
# ==========================================================

allowed_values = {
    "attack_vector": {
        "NETWORK",
        "ADJACENT",
        "ADJACENT_NETWORK",
        "LOCAL",
        "PHYSICAL",
        "UNKNOWN",
    },
    "attack_complexity": {
        "LOW",
        "HIGH",
        "UNKNOWN",
    },
    "privileges_required": {
        "NONE",
        "LOW",
        "HIGH",
        "UNKNOWN",
    },
    "user_interaction": {
        "NONE",
        "REQUIRED",
        "UNKNOWN",
    },
}


has_invalid_value = False


for column, allowed in allowed_values.items():
    actual_values = set(
        df[column].unique()
    )

    invalid = actual_values - allowed

    if invalid:
        has_invalid_value = True

        print(
            f"{column}에 예상하지 못한 값이 있습니다:",
            invalid,
        )


if not has_invalid_value:
    print("범주형 값 검사 완료: 이상 없음")

attack_complexity에 예상하지 못한 값이 있습니다: {'MEDIUM'}
privileges_required에 예상하지 못한 값이 있습니다: {'SINGLE'}


In [29]:
# ==========================================================
# 6. 기존 약 1만 건 데이터로 생성한 학습 객체 불러오기
# ==========================================================

model = joblib.load(
    "risk_model.pkl"
)

feature_data = joblib.load(
    "feature_data.pkl"
)

label_encoder = feature_data[
    "label_encoder"
]

tfidf_vectorizer = joblib.load(
    "tfidf_vectorizer.pkl"
)

feature_cols = joblib.load(
    "feature_cols.pkl"
)

encoded_columns = joblib.load(
    "encoded_columns.pkl"
)


print("기존 학습 객체 로드 완료")
print("모델 사용 범주형 컬럼:", feature_cols)
print("One-Hot 컬럼 개수:", len(encoded_columns))
print("모델 Feature 개수:", model.n_features_in_)

기존 학습 객체 로드 완료
모델 사용 범주형 컬럼: ['attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe']
One-Hot 컬럼 개수: 368
모델 Feature 개수: 668


In [30]:
# ==========================================================
# 7. 팀2 데이터를 학습 당시 Feature 구조로 변환
# ==========================================================

# 범주형 컬럼 One-Hot Encoding
encoded_features = pd.get_dummies(
    df[feature_cols],
    prefix=feature_cols,
    dtype=int,
)


print(
    "팀2 데이터 자체 One-Hot 컬럼 수:",
    encoded_features.shape[1],
)


# 학습 당시 One-Hot 컬럼과 정확히 동일하게 맞춤
encoded_features = encoded_features.reindex(
    columns=encoded_columns,
    fill_value=0,
)


print(
    "학습 구조로 맞춘 One-Hot 컬럼 수:",
    encoded_features.shape[1],
)


# Description은 반드시 transform만 사용
description_features = (
    tfidf_vectorizer.transform(
        df["description"]
    )
)


# 희소 행렬 변환
categorical_features = csr_matrix(
    encoded_features.values
)


# 범주형 + TF-IDF 결합
X_new = hstack(
    [
        categorical_features,
        description_features,
    ],
    format="csr",
)


print("팀2 예측 데이터 변환 완료")
print("입력 행 수:", X_new.shape[0])
print("입력 Feature 수:", X_new.shape[1])
print("모델 요구 Feature 수:", model.n_features_in_)


if X_new.shape[1] != model.n_features_in_:
    raise ValueError(
        "모델 Feature 개수와 팀2 변환 데이터의 Feature 개수가 다릅니다."
    )

팀2 데이터 자체 One-Hot 컬럼 수: 46
학습 구조로 맞춘 One-Hot 컬럼 수: 368
팀2 예측 데이터 변환 완료
입력 행 수: 121
입력 Feature 수: 668
모델 요구 Feature 수: 668


In [31]:
# ==========================================================
# 기존 약 1만 건 데이터로 학습한 모델 및 Feature 객체 불러오기
# ==========================================================

import joblib

model = joblib.load(
    "risk_model.pkl"
)

feature_data = joblib.load(
    "feature_data.pkl"
)

label_encoder = feature_data[
    "label_encoder"
]

tfidf_vectorizer = joblib.load(
    "tfidf_vectorizer.pkl"
)

feature_cols = joblib.load(
    "feature_cols.pkl"
)

encoded_columns = joblib.load(
    "encoded_columns.pkl"
)


print("학습 파일 불러오기 완료")
print("모델 종류:", type(model))
print("모델 사용 범주형 컬럼:", feature_cols)
print("학습 당시 One-Hot 컬럼 수:", len(encoded_columns))
print("모델 입력 Feature 수:", model.n_features_in_)

학습 파일 불러오기 완료
모델 종류: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
모델 사용 범주형 컬럼: ['attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe']
학습 당시 One-Hot 컬럼 수: 368
모델 입력 Feature 수: 668


In [32]:
# ==========================================================
# 팀2 JSON 데이터를 학습 당시 Feature 구조로 변환
# ==========================================================

from scipy.sparse import csr_matrix, hstack


# 1. 범주형 컬럼만 선택하여 One-Hot Encoding
encoded_features = pd.get_dummies(
    df[feature_cols],
    prefix=feature_cols,
    dtype=int,
)


print(
    "팀2 데이터에서 생성된 One-Hot 컬럼 수:",
    encoded_features.shape[1],
)


# 2. 학습 당시 One-Hot 컬럼 순서와 개수로 맞추기
encoded_features = encoded_features.reindex(
    columns=encoded_columns,
    fill_value=0,
)


print(
    "학습 구조로 맞춘 One-Hot 컬럼 수:",
    encoded_features.shape[1],
)


# 3. description을 기존 TF-IDF 객체로 변환
# 절대로 fit_transform을 사용하면 안 됨
description_features = (
    tfidf_vectorizer.transform(
        df["description"]
    )
)


# 4. 범주형 데이터를 희소 행렬로 변환
categorical_features = csr_matrix(
    encoded_features.values
)


# 5. 범주형 Feature + TF-IDF Feature 결합
X_new = hstack(
    [
        categorical_features,
        description_features,
    ],
    format="csr",
)


print("\n팀2 예측 데이터 Feature 변환 완료")
print("입력 데이터 행 수:", X_new.shape[0])
print("입력 Feature 수:", X_new.shape[1])
print("모델 요구 Feature 수:", model.n_features_in_)


# Feature 수 검증
if X_new.shape[1] != model.n_features_in_:
    raise ValueError(
        "모델이 요구하는 Feature 개수와 "
        "팀2 데이터의 Feature 개수가 다릅니다."
    )

팀2 데이터에서 생성된 One-Hot 컬럼 수: 46
학습 구조로 맞춘 One-Hot 컬럼 수: 368

팀2 예측 데이터 Feature 변환 완료
입력 데이터 행 수: 121
입력 Feature 수: 668
모델 요구 Feature 수: 668


In [33]:
# ==========================================================
# Random Forest 위험도 예측
# ==========================================================

prediction = model.predict(
    X_new
)


# 숫자로 인코딩된 예측값을 원래 위험도 문자로 변환
df["predicted_severity"] = (
    label_encoder.inverse_transform(
        prediction
    )
)


print("Random Forest 예측 완료")
print("예측 데이터 수:", len(df))


print("\n예측 위험도 분포")
print(
    df["predicted_severity"].value_counts()
)


preview_columns = [
    "cve_id",
    "cvss_score",
    "severity",
    "predicted_severity",
    "service",
    "version",
]

preview_columns = [
    column
    for column in preview_columns
    if column in df.columns
]


display(
    df[preview_columns].head(10)
)

Random Forest 예측 완료
예측 데이터 수: 121

예측 위험도 분포
predicted_severity
MEDIUM    89
HIGH      31
LOW        1
Name: count, dtype: int64


,cve_id,cvss_score,severity,predicted_severity,service,version
0,CVE-2011-2523,9.8,CRITICAL,MEDIUM,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,MEDIUM,ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,MEDIUM,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,MEDIUM,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,HIGH,ssh,4.7p1 Debian 8ubuntu1
5,CVE-2016-10012,7.8,HIGH,HIGH,ssh,4.7p1 Debian 8ubuntu1
6,CVE-2010-5107,7.5,HIGH,MEDIUM,ssh,4.7p1 Debian 8ubuntu1
7,CVE-2016-10708,7.5,HIGH,MEDIUM,ssh,4.7p1 Debian 8ubuntu1
8,CVE-2016-6515,7.5,HIGH,HIGH,ssh,4.7p1 Debian 8ubuntu1
9,CVE-2014-1692,7.3,HIGH,MEDIUM,ssh,4.7p1 Debian 8ubuntu1


In [34]:
# ==========================================================
# Random Forest 예측 신뢰도 계산
# ==========================================================

if hasattr(model, "predict_proba"):

    prediction_probabilities = (
        model.predict_proba(X_new)
    )

    df["prediction_confidence"] = (
        prediction_probabilities.max(axis=1)
        * 100
    ).round(2)

else:
    df["prediction_confidence"] = None


display(
    df[
        [
            "cve_id",
            "predicted_severity",
            "prediction_confidence",
        ]
    ].head(10)
)

,cve_id,predicted_severity,prediction_confidence
0,CVE-2011-2523,MEDIUM,36.56
1,CVE-2010-4478,MEDIUM,39.70
2,CVE-2016-1908,MEDIUM,35.84
3,CVE-2015-5600,MEDIUM,35.02
4,CVE-2015-8325,HIGH,44.42
5,CVE-2016-10012,HIGH,48.58
6,CVE-2010-5107,MEDIUM,39.41
7,CVE-2016-10708,MEDIUM,41.92
8,CVE-2016-6515,HIGH,38.62
9,CVE-2014-1692,MEDIUM,39.30


In [35]:
# ==========================================================
# 우선순위 점수 계산용 기준표
# ==========================================================

severity_score_map = {
    "LOW": 25,
    "MEDIUM": 50,
    "HIGH": 75,
    "CRITICAL": 100,
    "UNKNOWN": 50,
}

attack_vector_score_map = {
    "PHYSICAL": 20,
    "LOCAL": 40,
    "ADJACENT": 70,
    "ADJACENT_NETWORK": 70,
    "NETWORK": 100,
    "UNKNOWN": 50,
}

attack_complexity_score_map = {
    "HIGH": 40,
    "LOW": 100,
    "UNKNOWN": 60,
}

privileges_score_map = {
    "HIGH": 30,
    "LOW": 70,
    "NONE": 100,
    "UNKNOWN": 60,
}

user_interaction_score_map = {
    "REQUIRED": 40,
    "NONE": 100,
    "UNKNOWN": 60,
}


print("우선순위 점수 기준표 생성 완료")

우선순위 점수 기준표 생성 완료


In [36]:
# ==========================================================
# 우선순위 점수 및 대응 등급 함수
# ==========================================================

def calculate_priority_score(row) -> float:
    """
    CVSS 점수와 머신러닝 예측 위험도,
    공격 조건을 결합하여 내부 우선순위 점수를 계산한다.
    """

    cvss_score = float(
        row["cvss_score"]
    ) * 10

    severity_score = severity_score_map.get(
        row["predicted_severity"],
        50,
    )

    vector_score = attack_vector_score_map.get(
        row["attack_vector"],
        50,
    )

    complexity_score = (
        attack_complexity_score_map.get(
            row["attack_complexity"],
            60,
        )
    )

    privilege_score = privileges_score_map.get(
        row["privileges_required"],
        60,
    )

    interaction_score = (
        user_interaction_score_map.get(
            row["user_interaction"],
            60,
        )
    )

    score = (
        cvss_score * 0.50
        + severity_score * 0.20
        + vector_score * 0.10
        + complexity_score * 0.08
        + privilege_score * 0.07
        + interaction_score * 0.05
    )

    return round(score, 2)


def classify_response_priority(
    score: float,
) -> str:

    if score >= 85:
        return "긴급 대응"

    if score >= 70:
        return "우선 대응"

    if score >= 50:
        return "검토 필요"

    return "일반 관리"


print("우선순위 계산 함수 생성 완료")

우선순위 계산 함수 생성 완료


In [37]:
# ==========================================================
# 전체 취약점의 점수 및 대응 순위 계산
# ==========================================================

df["priority_score"] = df.apply(
    calculate_priority_score,
    axis=1,
)


df["response_priority"] = (
    df["priority_score"].apply(
        classify_response_priority
    )
)


# 위험도 점수와 CVSS가 높은 순서대로 정렬
df = df.sort_values(
    by=[
        "priority_score",
        "cvss_score",
    ],
    ascending=[
        False,
        False,
    ],
).reset_index(
    drop=True
)


# 순위 번호 부여
df["priority_rank"] = (
    df.index + 1
)


print("우선순위 계산 완료")
print("결과 데이터 수:", len(df))


result_columns = [
    "priority_rank",
    "cve_id",
    "service",
    "version",
    "cvss_score",
    "severity",
    "predicted_severity",
    "prediction_confidence",
    "priority_score",
    "response_priority",
]

result_columns = [
    column
    for column in result_columns
    if column in df.columns
]


display(
    df[result_columns].head(20)
)

우선순위 계산 완료
결과 데이터 수: 121


,priority_rank,cve_id,service,version,cvss_score,severity,predicted_severity,prediction_confidence,priority_score,response_priority
0,1,CVE-2008-0122,domain,9.4.2,10.0,HIGH,HIGH,51.71,90.2,긴급 대응
1,2,CVE-2010-0425,http,2.2.8,10.0,HIGH,HIGH,47.72,90.2,긴급 대응
2,3,CVE-2011-2523,ftp,2.3.4,9.8,CRITICAL,MEDIUM,36.56,84.2,우선 대응
3,4,CVE-2010-4478,ssh,4.7p1 Debian 8ubuntu1,9.8,CRITICAL,MEDIUM,39.70,84.2,우선 대응
4,5,CVE-2016-1908,ssh,4.7p1 Debian 8ubuntu1,9.8,CRITICAL,MEDIUM,35.84,84.2,우선 대응
5,6,CVE-2009-3555,http,2.2.8,9.8,CRITICAL,MEDIUM,40.42,84.2,우선 대응
6,7,CVE-2016-1286,domain,9.4.2,8.6,HIGH,HIGH,39.64,83.2,우선 대응
7,8,CVE-2012-3817,domain,9.4.2,7.8,HIGH,HIGH,39.46,79.2,우선 대응
8,9,CVE-2012-4244,domain,9.4.2,7.8,HIGH,HIGH,39.65,79.2,우선 대응
9,10,CVE-2012-5166,domain,9.4.2,7.8,HIGH,HIGH,39.11,79.2,우선 대응


In [38]:
# ==========================================================
# 팀4 및 챗봇 전달용 결과 컬럼 구성
# ==========================================================

output_columns = [
    "priority_rank",
    "cve_id",
    "service",
    "version",
    "cvss_score",
    "severity",
    "predicted_severity",
    "prediction_confidence",
    "priority_score",
    "response_priority",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
]


# 실제 존재하는 컬럼만 선택
output_columns = [
    column
    for column in output_columns
    if column in df.columns
]


priority_df = df[
    output_columns
].copy()


print("최종 출력 데이터 크기:", priority_df.shape)
print("최종 출력 컬럼:")
print(priority_df.columns.tolist())


display(
    priority_df.head(10)
)

최종 출력 데이터 크기: (121, 16)
최종 출력 컬럼:
['priority_rank', 'cve_id', 'service', 'version', 'cvss_score', 'severity', 'predicted_severity', 'prediction_confidence', 'priority_score', 'response_priority', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe', 'description']


,priority_rank,cve_id,service,version,cvss_score,severity,predicted_severity,prediction_confidence,priority_score,response_priority,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description
0,1,CVE-2008-0122,domain,9.4.2,10.0,HIGH,HIGH,51.71,90.2,긴급 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-189,Off-by-one error in the inet_network function ...
1,2,CVE-2010-0425,http,2.2.8,10.0,HIGH,HIGH,47.72,90.2,긴급 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,UNKNOWN,modules/arch/win32/mod_isapi.c in mod_isapi in...
2,3,CVE-2011-2523,ftp,2.3.4,9.8,CRITICAL,MEDIUM,36.56,84.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...
3,4,CVE-2010-4478,ssh,4.7p1 Debian 8ubuntu1,9.8,CRITICAL,MEDIUM,39.70,84.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable..."
4,5,CVE-2016-1908,ssh,4.7p1 Debian 8ubuntu1,9.8,CRITICAL,MEDIUM,35.84,84.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-287,The client in OpenSSH before 7.2 mishandles fa...
5,6,CVE-2009-3555,http,2.2.8,9.8,CRITICAL,MEDIUM,40.42,84.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-295,"The TLS protocol, and the SSL protocol 3.0 and..."
6,7,CVE-2016-1286,domain,9.4.2,8.6,HIGH,HIGH,39.64,83.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,UNKNOWN,named in ISC BIND 9.x before 9.9.8-P4 and 9.10...
7,8,CVE-2012-3817,domain,9.4.2,7.8,HIGH,HIGH,39.46,79.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-20,"ISC BIND 9.4.x, 9.5.x, 9.6.x, and 9.7.x before..."
8,9,CVE-2012-4244,domain,9.4.2,7.8,HIGH,HIGH,39.65,79.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,UNKNOWN,"ISC BIND 9.x before 9.7.6-P3, 9.8.x before 9.8..."
9,10,CVE-2012-5166,domain,9.4.2,7.8,HIGH,HIGH,39.11,79.2,우선 대응,NETWORK,LOW,UNKNOWN,UNKNOWN,CWE-189,"ISC BIND 9.x before 9.7.6-P4, 9.8.x before 9.8..."


In [39]:
# ==========================================================
# 팀4 전달용 파일 저장
# ==========================================================

import json


priority_df.to_csv(
    "priority.csv",
    index=False,
    encoding="utf-8-sig",
)


priority_output = priority_df.to_dict(
    orient="records"
)


with open(
    "priority.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        priority_output,
        file,
        ensure_ascii=False,
        indent=4,
    )


print("priority.csv 생성 완료")
print("priority.json 생성 완료")
print("저장된 결과 수:", len(priority_output))

priority.csv 생성 완료
priority.json 생성 완료
저장된 결과 수: 121
